In [2]:
from training_utilities_2nd_part import *

In [3]:
#oil
from variables_to_specify_oil import *

df, columns_to_normalize, oil_target_col, forecast_avg_target_col_name, avg_target_col_name, No_of_datapoints_in_one_day, start_date, end_date, delta, one_month_days, out_columns, oil_drop_columnss, oil_windows, index_of_one_month, one_month_window_size, date_col_name = variables_to_specify_oil()


from sklearn.preprocessing import MinMaxScaler

scaler = MinMaxScaler()

df[columns_to_normalize] = scaler.fit_transform(df[columns_to_normalize])

df = df.dropna().reset_index(drop=True)

convert_time(df, date_col_name)

oil_df = df
oil_time_steps = 1

# stationary

In [5]:
oil_len_of_training_data_of_stationary_model =7*No_of_datapoints_in_one_day

train = df[0:oil_len_of_training_data_of_stationary_model] 
test = df[oil_len_of_training_data_of_stationary_model:]

eval_df_first_month, stationary_model1 = stationary_model_with_hptuning(train, test, one_month_window_size, 2, out_columns, oil_target_col, oil_drop_columnss)

sum_training_time_stat1 = eval_df_first_month['training_time'].sum()
print('sum_training_time is: ', sum_training_time_stat1)

print(eval_df_first_month['Testing Error'].mean())

Model Type: XGBRegressor
Storage Required: 0.10 MB
model storage is : 0.10306930541992188


total_time is:  0.2173778749999995
sum_training_time is:  2.985804249999994
0.14257475945176634


# Model reuse

In [6]:
# Model reuse
daily_df_avg = get_elect_daily_avg(oil_df, No_of_datapoints_in_one_day, oil_target_col, avg_target_col_name)


seasonality_periods_acf_ls, seasonality_periods_acf, segmented_daily_df_avg, filtered_most_similar_dict_wass, filtered_most_similar_dict_tvd, forecast_daily_df_avg, segmented_forecast_daily_df_avg, filtered_forecasted_most_similar_dict_wass, filtered_forecasted_most_similar_dict_tvd = get_seasonality_segments_and_similarities(daily_df_avg, avg_target_col_name, forecast_avg_target_col_name, 7)

Detected seasonality periods (ACF): [  7 297 309 314 317 327 330 341 345 351 359]
median_value is:  327


## drift detection

In [7]:
df_copy = oil_df[[oil_target_col]]
target_col = oil_target_col
time_steps = oil_time_steps

df_copy['date'] = pd.to_datetime(df_copy.index)
multiplier = No_of_datapoints_in_one_day
x = 7* multiplier
window_len_=[x]
drift_results_df_ls = []
for i in window_len_:
    start_drift_detection_time = timeit.default_timer()
    drift_results_df = detect_drift_univariate(
        df_copy,
        target_col=oil_target_col,
        window_lengths=window_len_,
        arima_order=(1, 0, 0)
    )
    drift_results_df_ls.append(drift_results_df)
    drift_detection_time = timeit.default_timer() - start_drift_detection_time
    num_true = drift_results_df['drift_detected'].sum()
    print("i is: ", i, " and the Number of True values in 'drift_detected':", num_true, " total number of rows are : ", len(drift_results_df))
    print("drift detection time is: ", drift_detection_time)
    drift_results_df = drift_results_df_ls[0]
    drift_indices = list(drift_results_df.index[drift_results_df['drift_detected']])
    print("indices are: ", drift_indices)

Fold 0: Train size=33, Test size=33
Fold 1: Train size=66, Test size=33
Fold 2: Train size=99, Test size=33
Fold 3: Train size=132, Test size=33
Skipping fold 4: Insufficient training or test data.
Fold 0: Train size=33, Test size=33
Fold 1: Train size=66, Test size=33
Fold 2: Train size=99, Test size=33
Fold 3: Train size=132, Test size=33
Skipping fold 4: Insufficient training or test data.
Fold 0: Train size=33, Test size=33
Fold 1: Train size=66, Test size=33
Fold 2: Train size=99, Test size=33
Fold 3: Train size=132, Test size=33
Skipping fold 4: Insufficient training or test data.
Fold 0: Train size=33, Test size=33
Fold 1: Train size=66, Test size=33
Fold 2: Train size=99, Test size=33
Fold 3: Train size=132, Test size=33
Skipping fold 4: Insufficient training or test data.
Fold 0: Train size=33, Test size=33
Fold 1: Train size=66, Test size=33
Fold 2: Train size=99, Test size=33
Fold 3: Train size=132, Test size=33
Skipping fold 4: Insufficient training or test data.
Fold 0: Tr

In [8]:
eval_df_monthly2, avg_ml_storage1 = new_copied_reuse_with_hptuning_no_while_loop_with_drift(filtered_most_similar_dict_wass, stationary_model1, oil_len_of_training_data_of_stationary_model,oil_df, "SA", oil_target_col, oil_drop_columnss, oil_time_steps, seasonality_periods_acf, No_of_datapoints_in_one_day, drift_indices, 2)


window is:  168
i/window is :  1.0
Model Type: XGBRegressor
Storage Required: 0.10 MB


window is:  336
i/window is :  2.0
Model Type: XGBRegressor
Storage Required: 0.09 MB


window is:  504
i/window is :  3.0
Model Type: XGBRegressor
Storage Required: 0.10 MB


window is:  672
i/window is :  4.0
Model Type: XGBRegressor
Storage Required: 0.10 MB


window is:  840
i/window is :  5.0
similar_month_index is :  2
month_index:  5




window is:  1008
i/window is :  6.0
similar_month_index is :  2
month_index:  6




window is:  1176
i/window is :  7.0
similar_month_index is :  1
month_index:  7




window is:  1344
i/window is :  8.0
similar_month_index is :  0
month_index:  8




window is:  1512
i/window is :  9.0
Model Type: XGBRegressor
Storage Required: 0.07 MB


window is:  1680
i/window is :  10.0
Model Type: XGBRegressor
Storage Required: 0.09 MB


window is:  1848
i/window is :  11.0
Model Type: XGBRegressor
Storage Required: 0.09 MB


window is:  2016
i/window is :  12.0
Model T

In [9]:
eval_df_monthly2, avg_ml_storage2 = new_copied_reuse_with_hptuning_no_while_loop_with_drift(filtered_most_similar_dict_tvd, stationary_model1, oil_len_of_training_data_of_stationary_model,oil_df, "SA", oil_target_col, oil_drop_columnss, oil_time_steps, seasonality_periods_acf, No_of_datapoints_in_one_day, drift_indices, 2)

window is:  168
i/window is :  1.0
Model Type: XGBRegressor
Storage Required: 0.10 MB


window is:  336
i/window is :  2.0
Model Type: XGBRegressor
Storage Required: 0.09 MB


window is:  504
i/window is :  3.0
Model Type: XGBRegressor
Storage Required: 0.10 MB


window is:  672
i/window is :  4.0
Model Type: XGBRegressor
Storage Required: 0.10 MB


window is:  840
i/window is :  5.0
Model Type: XGBRegressor
Storage Required: 0.08 MB


window is:  1008
i/window is :  6.0
similar_month_index is :  3
month_index:  6




window is:  1176
i/window is :  7.0
similar_month_index is :  5
month_index:  7




window is:  1344
i/window is :  8.0
similar_month_index is :  6
previous_model_i is :  1344
math.floor(previous_model_i/window) is:  8
len(models_ls) is: 7
Model Type: XGBRegressor
Storage Required: 0.07 MB


window is:  1512
i/window is :  9.0
similar_month_index is :  4
month_index:  9




window is:  1680
i/window is :  10.0
similar_month_index is :  5
month_index:  10




window is:  1

In [10]:
eval_df_monthly2, avg_ml_storage3 = new_copied_reuse_with_hptuning_no_while_loop_with_drift(filtered_forecasted_most_similar_dict_wass, stationary_model1, oil_len_of_training_data_of_stationary_model,oil_df, "ES", oil_target_col, oil_drop_columnss, oil_time_steps, seasonality_periods_acf, No_of_datapoints_in_one_day, drift_indices, 2)

window is:  168
Model Type: XGBRegressor
Storage Required: 0.10 MB


window is:  336
Model Type: XGBRegressor
Storage Required: 0.09 MB


window is:  504
Model Type: XGBRegressor
Storage Required: 0.10 MB


window is:  672
Model Type: XGBRegressor
Storage Required: 0.10 MB


window is:  840
Model Type: XGBRegressor
Storage Required: 0.08 MB


window is:  1008
similar_month_index is :  1
month_index:  5



window is:  1176
Model Type: XGBRegressor
Storage Required: 0.08 MB


window is:  1344
similar_month_index is :  5
previous_model_i is :  1344
math.floor(previous_model_i/window) is:  8
len(models_ls) is: 7
Model Type: XGBRegressor
Storage Required: 0.07 MB


window is:  1512
similar_month_index is :  6
month_index:  8




window is:  1680
Model Type: XGBRegressor
Storage Required: 0.09 MB


window is:  1848
Model Type: XGBRegressor
Storage Required: 0.09 MB


window is:  2016
Model Type: XGBRegressor
Storage Required: 0.09 MB


window is:  2184
similar_month_index is :  10
month_inde

In [11]:
eval_df_monthly2, avg_ml_storage4 = new_copied_reuse_with_hptuning_no_while_loop_with_drift(filtered_forecasted_most_similar_dict_tvd, stationary_model1, oil_len_of_training_data_of_stationary_model,oil_df, "ES", oil_target_col, oil_drop_columnss, oil_time_steps, seasonality_periods_acf, No_of_datapoints_in_one_day, drift_indices, 2)

window is:  168
Model Type: XGBRegressor
Storage Required: 0.10 MB


window is:  336
Model Type: XGBRegressor
Storage Required: 0.09 MB


window is:  504
Model Type: XGBRegressor
Storage Required: 0.10 MB


window is:  672
Model Type: XGBRegressor
Storage Required: 0.10 MB


window is:  840
Model Type: XGBRegressor
Storage Required: 0.08 MB


window is:  1008
similar_month_index is :  3
month_index:  5




window is:  1176
Model Type: XGBRegressor
Storage Required: 0.08 MB


window is:  1344
Model Type: XGBRegressor
Storage Required: 0.09 MB


window is:  1512
Model Type: XGBRegressor
Storage Required: 0.07 MB


window is:  1680
Model Type: XGBRegressor
Storage Required: 0.09 MB


window is:  1848
Model Type: XGBRegressor
Storage Required: 0.09 MB


window is:  2016
Model Type: XGBRegressor
Storage Required: 0.09 MB


window is:  2184
Model Type: XGBRegressor
Storage Required: 0.09 MB


window is:  2352
Model Type: XGBRegressor
Storage Required: 0.10 MB


window is:  2520
Model Type: X

In [12]:
avg_ml_storage_reuse = (avg_ml_storage1+avg_ml_storage2+avg_ml_storage3+avg_ml_storage4)/4
print(avg_ml_storage_reuse)

0.08862255020738469


# informed

In [14]:
informed_update(stationary_model1,oil_df, target_col, oil_drop_columnss,time_steps, seasonality_periods_acf,No_of_datapoints_in_one_day, drift_indices,2)

window is:  168
Model Type: XGBRegressor
Storage Required: 0.10 MB
window is:  336
Model Type: XGBRegressor
Storage Required: 0.09 MB
window is:  504
Model Type: XGBRegressor
Storage Required: 0.10 MB
window is:  672
window is:  840
window is:  1008
window is:  1176
window is:  1344
window is:  1512
window is:  1680
window is:  1848
Model Type: XGBRegressor
Storage Required: 0.09 MB
window is:  2016
Model Type: XGBRegressor
Storage Required: 0.09 MB
window is:  2184
Model Type: XGBRegressor
Storage Required: 0.09 MB
window is:  2352
window is:  2520
Model Type: XGBRegressor
Storage Required: 0.09 MB
window is:  2688
window is:  2856
Model Type: XGBRegressor
Storage Required: 0.09 MB
window is:  3024
window is:  3192
Model Type: XGBRegressor
Storage Required: 0.10 MB
window is:  3360
Model Type: XGBRegressor
Storage Required: 0.08 MB
window is:  3528
window is:  3696
window is:  3864
window is:  4032
Model Type: XGBRegressor
Storage Required: 0.09 MB
window is:  4200
Model Type: XGBRegr

# periodical

In [8]:
periodical_retraining_with_hptuning(2, oil_df, oil_windows, out_columns, oil_target_col, oil_drop_columnss)

window is : 120
window size is :  120
Model Type: XGBRegressor
Storage Required: 0.07 MB
Model Type: XGBRegressor
Storage Required: 0.06 MB
Model Type: XGBRegressor
Storage Required: 0.07 MB
Model Type: XGBRegressor
Storage Required: 0.06 MB
Model Type: XGBRegressor
Storage Required: 0.07 MB
Model Type: XGBRegressor
Storage Required: 0.07 MB
Model Type: XGBRegressor
Storage Required: 0.06 MB
Model Type: XGBRegressor
Storage Required: 0.06 MB
Model Type: XGBRegressor
Storage Required: 0.06 MB
Model Type: XGBRegressor
Storage Required: 0.06 MB
Model Type: XGBRegressor
Storage Required: 0.06 MB
Model Type: XGBRegressor
Storage Required: 0.06 MB
Model Type: XGBRegressor
Storage Required: 0.05 MB
Model Type: XGBRegressor
Storage Required: 0.06 MB
Model Type: XGBRegressor
Storage Required: 0.06 MB
Model Type: XGBRegressor
Storage Required: 0.06 MB
Model Type: XGBRegressor
Storage Required: 0.06 MB
Model Type: XGBRegressor
Storage Required: 0.05 MB
Model Type: XGBRegressor
Storage Required: 0

([          Training dataset     Testing dataset       mae       mse      rmse  \
  0    trained on window i-1  tested on window i  0.036827  0.002602  0.051011   
  1    trained on window i-1  tested on window i  0.091881  0.011677  0.108062   
  2    trained on window i-1  tested on window i  0.183393  0.038222  0.195505   
  3    trained on window i-1  tested on window i  0.070127  0.007027  0.083826   
  4    trained on window i-1  tested on window i  0.035127  0.002020  0.044946   
  ..                     ...                 ...       ...       ...       ...   
  139  trained on window i-1  tested on window i  0.019452  0.000548  0.023406   
  140  trained on window i-1  tested on window i  0.027648  0.001348  0.036717   
  141  trained on window i-1  tested on window i  0.042343  0.002343  0.048400   
  142  trained on window i-1  tested on window i  0.035870  0.002171  0.046599   
  143  trained on window i-1  tested on window i  0.031133  0.001613  0.040167   
  
             